# Tutorial 02 — Tactics and Agentic Systems

A `Tactic` is the **program** in lllm. Where `Agent` is a single LLM identity, `Tactic` is the orchestration layer that wires agents together, manages control flow, and tracks execution.

| Section | Concepts covered |
|---------|------------------|
| 0 | Environment setup |
| 1 | What is a `Tactic` and why it exists |
| 2 | Minimal tactic: `name`, `agent_group`, `call()` |
| 3 | Configuring a tactic (Python dict and YAML) |
| 4 | Session tracking: `TacticCallSession` |
| 5 | Multi-agent sequential pipeline |
| 6 | Sub-tactic composition |
| 7 | Tactic inheritance and unregistered base classes |
| 8 | Dialog forking inside a tactic |
| 9 | Batch and async execution: `bcall`, `acall`, `ccall` |
| 10 | Logging with `LogStore` |
| 11 | Tagging sessions and filtering |
| 12 | Cost aggregation and debugging failures |

---
## Section 0 — Environment Setup

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env", override=False)

has_openai = bool(os.getenv("OPENAI_API_KEY"))
has_anthropic = bool(os.getenv("ANTHROPIC_API_KEY"))

if not (has_openai or has_anthropic):
    raise EnvironmentError(
        "No API key found. Set OPENAI_API_KEY or ANTHROPIC_API_KEY in your .env file."
    )

DEFAULT_MODEL = "gpt-4o-mini" if has_openai else "claude-haiku-4-5-20251001"
SMART_MODEL   = "gpt-4o"      if has_openai else "claude-sonnet-4-6"
print(f"Default model : {DEFAULT_MODEL}")
print(f"Smart model   : {SMART_MODEL}")

In [ ]:
import sys, asyncio, tempfile, os, json
sys.path.insert(0, "..")

from lllm import Agent, Prompt, Tactic
from lllm.invokers import build_invoker
from helpers.mock_data import SAMPLE_REVIEWS, SAMPLE_TASKS
from helpers.display import print_section, print_response

---
## Section 1 — What is a `Tactic`?

Think of the relationship like this:

```
Agent   = a single LLM identity  (the brain)
Dialog  = the agent's memory     (the mental state)
Tactic  = the program            (the orchestration logic)
```

A `Tactic` is modeled after PyTorch's `nn.Module`:

- **Composable** — tactics can call other tactics
- **Stateless between calls** — each call gets fresh agent instances
- **Observable** — every call produces a `TacticCallSession` with cost, errors, traces

Every `Tactic` subclass must define:
1. `name` — unique string identifier (used in logs and the tactic registry)
2. `agent_group` — list of agent names the tactic needs
3. `call(task, **kwargs)` — the orchestration logic

---
## Section 2 — Minimal Tactic

The simplest possible tactic has one agent and a straight `call()` body.

In [ ]:
class SummarizerTactic(Tactic):
    name = "summarizer"
    agent_group = ["writer"]   # list of agent names this tactic requires

    def call(self, task: str, **kwargs) -> str:
        # self.agents is a dict of ready-to-use Agent instances
        writer = self.agents["writer"]

        writer.open("session")
        writer.receive(f"Summarize this text in 3 bullet points:\n\n{task}")
        response = writer.respond()

        return response.content


print("Tactic defined:", SummarizerTactic.name)

---
## Section 3 — Configuring a Tactic

Tactics receive a **config dict** at construction time. This is where you declare models, system prompts, and per-agent overrides.

The `global` block sets defaults for all agents.  
`agent_configs` lists per-agent overrides that are **deep-merged on top** of the global settings.

In [ ]:
config = {
    "tactic_type": "summarizer",   # name of the registered Tactic subclass
    "global": {
        "model_name": DEFAULT_MODEL,
        "model_args": {"temperature": 0.3},
    },
    "agent_configs": [
        {
            "name": "writer",
            "system_prompt": "You are a concise technical writer. Use bullet points.",
            "model_args": {"max_completion_tokens": 512},
        },
    ],
}

tactic = SummarizerTactic(config)

# Calling the tactic — parentheses invoke _execute() which wraps call()
text = """
Machine learning models have become increasingly powerful, enabling applications
ranging from natural language processing to image recognition. However, they require
large amounts of training data and significant computational resources. Transfer
learning has emerged as a key technique to reduce these requirements by reusing
pre-trained models as starting points for new tasks.
"""

result = tactic(text.strip())
print_section("Summarizer Output")
print(result)

---
## Section 4 — Session Tracking

Every tactic call automatically records a `TacticCallSession`.  
Pass `return_session=True` to get it back alongside the result.

In [ ]:
session = tactic(text.strip(), return_session=True)

print("Type           :", type(session))
print("State          :", session.state)              # 'success' or 'failure'
print("Result         :", str(session.delivery)[:80]) # the value returned by call()
print("Total cost     :", session.total_cost)
print("Agent call cnt :", session.agent_call_count)

In [ ]:
# Detailed per-agent breakdown
for agent_name, calls in session.agent_sessions.items():
    print(f"\nAgent: {agent_name}")
    for i, call in enumerate(calls):
        print(f"  Call {i+1}:")
        print(f"    Tool interrupts     : {len(call.interrupts)}")
        print(f"    Exception retries   : {call.exception_retries_count}")
        print(f"    Cost                : {call.cost}")

In [ ]:
# Compact summary dict
print(json.dumps(session.summary(), indent=2, default=str))

---
## Section 5 — Multi-Agent Sequential Pipeline

The most common pattern: Agent A processes the input and feeds the result to Agent B.  
Each agent does one thing well; the tactic wires them together.

In [ ]:
class ResearchTactic(Tactic):
    """Two-agent pipeline: fact-gatherer → synthesizer."""
    name = "researcher"
    agent_group = ["finder", "synthesizer"]

    def call(self, task: str, **kwargs) -> str:
        finder      = self.agents["finder"]
        synthesizer = self.agents["synthesizer"]

        # Stage 1: gather raw facts
        finder.open("gather")
        finder.receive(f"List 5 specific, concrete facts about: {task}")
        facts = finder.respond().content

        # Stage 2: synthesize into a coherent paragraph
        synthesizer.open("synthesize")
        synthesizer.receive(
            f"Here are the raw facts:\n\n{facts}\n\n"
            "Write a clear, coherent paragraph that weaves these facts together naturally."
        )
        return synthesizer.respond().content


research_config = {
    "tactic_type": "researcher",
    "global": {
        "model_name": DEFAULT_MODEL,
        "model_args": {"temperature": 0.2},
    },
    "agent_configs": [
        {
            "name": "finder",
            "system_prompt": (
                "You are a fact researcher. List concrete, specific facts. "
                "No opinions, no speculation."
            ),
        },
        {
            "name": "synthesizer",
            "system_prompt": (
                "You are a technical writer. Transform bullet-point facts into "
                "a well-structured, readable paragraph."
            ),
        },
    ],
}

research_tactic = ResearchTactic(research_config)

result = research_tactic("the Python GIL (Global Interpreter Lock)")
print_section("Research Result")
print(result)

In [ ]:
# Three-agent pipeline: planner → executor → reviewer
class WritingPipelineTactic(Tactic):
    """outline → draft → editorial review."""
    name = "writing_pipeline"
    agent_group = ["outliner", "writer", "editor"]

    def call(self, task: str, **kwargs) -> dict:
        outliner = self.agents["outliner"]
        writer   = self.agents["writer"]
        editor   = self.agents["editor"]

        # Stage 1: outline
        outliner.open("outline")
        outliner.receive(f"Create a 4-point outline for: {task}")
        outline = outliner.respond().content

        # Stage 2: draft
        writer.open("write")
        writer.receive(
            f"Expand this outline into a short 3-paragraph article:\n\n{outline}"
        )
        draft = writer.respond().content

        # Stage 3: editorial pass
        editor.open("edit")
        editor.receive(
            f"Edit this draft for clarity and concision. Return only the improved text:\n\n{draft}"
        )
        final = editor.respond().content

        return {"outline": outline, "draft": draft, "final": final}


writing_config = {
    "global": {
        "model_name": DEFAULT_MODEL,
        "model_args": {"temperature": 0.6},
    },
    "agent_configs": [
        {"name": "outliner", "system_prompt": "You create concise, numbered outlines."},
        {"name": "writer",   "system_prompt": "You write clear, engaging articles."},
        {"name": "editor",   "system_prompt": "You polish prose. Cut filler words. Improve flow."},
    ],
}

writing_tactic = WritingPipelineTactic(writing_config)

output = writing_tactic("why Python became the dominant language for data science")
print_section("Outline")
print(output["outline"])
print_section("Final Article")
print(output["final"])

---
## Section 6 — Sub-Tactic Composition

Tactics can call other tactics, just like functions calling functions.  
Sub-tactics assigned as attributes (`self.sub = OtherTactic(config)`) are automatically registered in `self.sub_tactics` and their session costs are **folded into the parent session** for aggregation.

In [ ]:
class PipelineTactic(Tactic):
    """Research first, then write an article based on the research."""
    name = "pipeline"
    agent_group = ["coordinator"]

    def __init__(self, config, **kwargs):
        super().__init__(config, **kwargs)
        # Sub-tactic is declared here; sessions are tracked automatically
        self.researcher = ResearchTactic(research_config)

    def call(self, task: str, **kwargs) -> str:
        # Step 1: use the sub-tactic to gather facts
        facts = self.researcher(task)   # this is itself a full tactic call

        # Step 2: coordinate a final synthesized response
        coordinator = self.agents["coordinator"]
        coordinator.open("plan")
        coordinator.receive(
            f"Based on this research:\n{facts}\n\n"
            "Write a one-paragraph executive summary with an action recommendation."
        )
        return coordinator.respond().content


pipeline_config = {
    "global": {"model_name": DEFAULT_MODEL},
    "agent_configs": [
        {
            "name": "coordinator",
            "system_prompt": "You are a senior analyst. Write crisp executive summaries.",
        },
    ],
}

pipeline = PipelineTactic(pipeline_config)

session = pipeline("async programming in Python", return_session=True)
print_section("Pipeline Output")
print(session.delivery)

print("\nSub-tactic sessions:", list(session.sub_tactic_sessions.keys()))
print("Total cost (all agents):", session.total_cost)

---
## Section 7 — Tactic Inheritance and Unregistered Base Classes

Subclass any existing `Tactic` to **extend its logic** without rewriting it.  
Use `register=False` for base classes that should not appear in the registry.

In [ ]:
# Unregistered base — shared helpers, never instantiated directly
class BaseWritingTactic(Tactic, register=False):
    """Provides a shared _polish() helper. Not directly instantiable from config."""
    agent_group = ["writer"]

    def _polish(self, text: str) -> str:
        """Internal: improve prose clarity."""
        writer = self.agents["writer"]
        writer.open("polish")
        writer.receive(
            f"Rewrite this text to be clearer and more concise. Keep it factual:\n\n{text}"
        )
        return writer.respond().content


# Blog post tactic — adds SEO review on top of the polish step
class BlogPostTactic(BaseWritingTactic):
    name = "blog_post"
    agent_group = ["writer", "seo_reviewer"]   # extends the parent's agent_group

    def call(self, task: str, **kwargs) -> str:
        polished = self._polish(task)   # inherited from BaseWritingTactic

        seo = self.agents["seo_reviewer"]
        seo.open("seo")
        seo.receive(
            f"Add two or three relevant SEO keywords naturally into this text. "
            f"Return only the revised text:\n\n{polished}"
        )
        return seo.respond().content


# Technical doc tactic — adds a code example step
class TechDocTactic(BaseWritingTactic):
    name = "tech_doc"
    agent_group = ["writer", "code_reviewer"]

    def call(self, task: str, **kwargs) -> str:
        polished = self._polish(task)   # same inherited helper

        coder = self.agents["code_reviewer"]
        coder.open("code")
        coder.receive(
            f"Add a short, relevant Python code example to this technical explanation. "
            f"Use a ```python block:\n\n{polished}"
        )
        return coder.respond().content


shared_agent_config = {
    "global": {"model_name": DEFAULT_MODEL, "model_args": {"temperature": 0.4}},
    "agent_configs": [
        {"name": "writer",       "system_prompt": "You are a skilled writer. Improve prose quality."},
        {"name": "seo_reviewer", "system_prompt": "You add SEO keywords naturally."},
        {"name": "code_reviewer","system_prompt": "You add clear, minimal Python code examples."},
    ],
}

raw_text = (
    "Python decorators are a way to modify functions. They wrap a function "
    "and can add behavior before or after it runs."
)

blog_tactic = BlogPostTactic(shared_agent_config)
tech_tactic = TechDocTactic(shared_agent_config)

print_section("Blog Post (SEO-optimized)")
print(blog_tactic(raw_text)[:500])

print_section("Tech Doc (with code example)")
print(tech_tactic(raw_text)[:600])

In [ ]:
# Extend an existing concrete tactic — override call() and call super()
class EditedWritingPipeline(WritingPipelineTactic):
    """Adds a fact-checker stage after the editor."""
    name = "edited_writing_pipeline"
    agent_group = ["outliner", "writer", "editor", "fact_checker"]

    def call(self, task: str, **kwargs) -> dict:
        # Reuse the parent's three-stage pipeline
        output = super().call(task, **kwargs)

        # Add a fact-checking pass
        checker = self.agents["fact_checker"]
        checker.open("check")
        checker.receive(
            f"Review this article for factual accuracy. Flag any suspicious claims:\n\n"
            f"{output['final']}"
        )
        fact_report = checker.respond().content

        return {**output, "fact_report": fact_report}


extended_config = {
    **writing_config,
    "agent_configs": writing_config["agent_configs"] + [
        {"name": "fact_checker", "system_prompt": "You are a fact-checker. Be skeptical."}
    ],
}

extended_tactic = EditedWritingPipeline(extended_config)
output = extended_tactic("the history of the Python programming language")

print_section("Final Article")
print(output["final"][:400])
print_section("Fact-Check Report")
print(output["fact_report"][:400])

---
## Section 8 — Dialog Forking Inside a Tactic

Forking is powerful inside tactics: establish shared context once, then branch to explore multiple hypotheses in parallel (or sequence), without any branch polluting the others.

In [ ]:
class HypothesisTactic(Tactic):
    """
    Given a problem and a list of hypotheses:
    1. Establish shared context with the analyst agent
    2. Fork the dialog for each hypothesis (independent evaluation)
    3. Synthesize all evaluations in the main dialog
    """
    name = "hypothesis_explorer"
    agent_group = ["analyst"]

    def call(self, task: str, hypotheses: list[str], **kwargs) -> str:
        analyst = self.agents["analyst"]

        # Establish shared context
        analyst.open("base")
        analyst.receive(f"You are about to evaluate competing hypotheses about: {task}")
        analyst.respond()  # model acknowledges context

        # Evaluate each hypothesis in a separate fork
        evaluations = {}
        for i, hypothesis in enumerate(hypotheses):
            branch = f"branch_{i}"
            analyst.fork("base", branch)
            analyst.receive(f"Evaluate this hypothesis in 2-3 sentences: {hypothesis}")
            evaluations[hypothesis] = analyst.respond().content
            analyst.close(branch)  # clean up

        # Back to the main thread — synthesize all evaluations
        analyst.switch("base")
        summary = "\n".join(
            f"- Hypothesis: {h}\n  Evaluation: {e[:120]}..."
            for h, e in evaluations.items()
        )
        analyst.receive(
            f"Given these evaluations:\n{summary}\n\n"
            "Which hypothesis is strongest and why? Answer in 2-3 sentences."
        )
        return analyst.respond().content


hypothesis_config = {
    "global": {"model_name": DEFAULT_MODEL, "model_args": {"temperature": 0.3}},
    "agent_configs": [
        {"name": "analyst", "system_prompt": "You are a critical analyst. Evaluate ideas rigorously."},
    ],
}

explorer = HypothesisTactic(hypothesis_config)

result = explorer(
    task="why Python is dominant in data science",
    hypotheses=[
        "Python won because of NumPy and SciPy being developed early.",
        "Python won because of its readable syntax lowering the barrier for scientists.",
        "Python won because Google and major tech firms adopted it for internal tools.",
    ],
)

print_section("Strongest Hypothesis")
print(result)

---
## Section 9 — Batch and Async Execution

lllm provides three concurrency modes for running many tasks:

| Method | Description |
|--------|-------------|
| `bcall(tasks, max_workers=N)` | Thread pool — synchronous, returns list of results |
| `acall(task)` | Single async call (for use inside `async` frameworks) |
| `ccall(tasks, max_workers=N)` | Async generator — yields `(index, result)` as tasks complete |

Each task gets **fresh agent instances** — no shared state, no concurrency collisions.

In [ ]:
import time

class ClassifierTactic(Tactic):
    """Classify a text snippet into a category."""
    name = "classifier"
    agent_group = ["classifier"]

    def call(self, task: str, **kwargs) -> str:
        agent = self.agents["classifier"]
        agent.open("classify")
        agent.receive(task)
        return agent.respond().content


classifier_config = {
    "global": {"model_name": DEFAULT_MODEL, "model_args": {"temperature": 0.0}},
    "agent_configs": [
        {
            "name": "classifier",
            "system_prompt": (
                "You are a text classifier. Respond with exactly one of: "
                "TECHNICAL, BILLING, GENERAL. Nothing else."
            ),
        },
    ],
}

classifier_tactic = ClassifierTactic(classifier_config)

tasks_to_classify = [
    "My app crashes when I upload files larger than 100 MB.",
    "I was charged twice this month.",
    "What are your opening hours?",
    "The API rate limit error is 429 but I'm only sending 5 req/s.",
    "How do I cancel my subscription?",
    "Is there a discount for annual plans?",
    "Database connection timeout after 30 seconds.",
    "Where can I find my invoice?",
]

In [ ]:
# --- 9a. bcall — synchronous thread pool ---
start = time.perf_counter()
results = classifier_tactic.bcall(tasks_to_classify, max_workers=4)
elapsed = time.perf_counter() - start

print(f"bcall: {len(results)} tasks in {elapsed:.2f}s\n")
for task, result in zip(tasks_to_classify, results):
    print(f"  [{result.strip():10}] {task[:60]}")

In [ ]:
# --- 9b. fail_fast vs collecting errors ---
# With fail_fast=False, failed tasks return the exception instead of raising
results_safe = classifier_tactic.bcall(
    tasks_to_classify,
    max_workers=4,
    fail_fast=False,
)

for i, r in enumerate(results_safe):
    if isinstance(r, Exception):
        print(f"  Task {i} FAILED: {r}")
    else:
        print(f"  Task {i} OK: {r.strip()}")

In [ ]:
# --- 9c. acall — single async call ---
async def run_single():
    result = await classifier_tactic.acall("My payment failed.")
    return result

# In a notebook, use asyncio.run or await directly depending on the environment
result = asyncio.run(run_single())
print("acall result:", result)

In [ ]:
# --- 9d. ccall — async generator, yields results as they complete ---
async def run_ccall():
    results = {}
    async for idx, result in classifier_tactic.ccall(tasks_to_classify, max_workers=4):
        results[idx] = result.strip()
        print(f"  Completed task {idx}: {result.strip()}")
    return results

ordered = asyncio.run(run_ccall())
print(f"\nAll {len(ordered)} results collected")

---
## Section 10 — Logging with `LogStore`

Every tactic call can be persisted to a `LogStore`. This gives you:
- A full audit trail of every agent call
- Token counts and dollar costs per session
- Error tracebacks for failed calls
- Queryable history for analytics

Three backends are available:

| Backend | Best for |
|---------|----------|
| `LocalFileBackend` | Development, small projects |
| `SQLiteBackend` | Production, queryable store |
| `NoOpBackend` | Tests, benchmarks |


In [ ]:
from lllm import LogStore, LocalFileBackend, SQLiteBackend, NoOpBackend
from lllm.logging import local_store, sqlite_store, noop_store

# Create a temporary SQLite store for this tutorial
db_path = os.path.join(tempfile.gettempdir(), "lllm_tutorial_02.db")
store = sqlite_store(db_path)
print(f"Logging to: {db_path}")

# Attach the log store to the tactic at construction time
logged_tactic = ClassifierTactic(classifier_config, log_store=store)

In [ ]:
# Run several tasks — each is automatically persisted
for task in tasks_to_classify[:4]:
    logged_tactic(task)

print("Ran 4 tasks")

---
## Section 11 — Tagging Sessions and Filtering

Pass `tags` to any tactic call to annotate it. Tags are stored in the index and support efficient filtering — no need to load full session data.

In [ ]:
# Tag calls by environment, experiment ID, user, etc.
logged_tactic(
    "The API returns 500 on large payloads.",
    tags={"env": "staging", "experiment": "v2", "user": "alice"},
)
logged_tactic(
    "I need a refund.",
    tags={"env": "prod", "experiment": "v2", "user": "bob"},
)
logged_tactic(
    "How do I reset my password?",
    tags={"env": "prod", "experiment": "v1", "user": "carol"},
)

print("Tagged calls recorded")

In [ ]:
# List all sessions
all_sessions = store.list_sessions(limit=20)
print(f"Total sessions: {len(all_sessions)}\n")
for s in all_sessions:
    print(f"  {s.session_id[:12]}...  state={s.state}  cost={s.total_cost}")

In [ ]:
# Filter by tag
prod_sessions = store.list_sessions(tags={"env": "prod"})
print(f"Prod sessions: {len(prod_sessions)}")

v2_sessions = store.list_sessions(tags={"experiment": "v2"})
print(f"Experiment v2 sessions: {len(v2_sessions)}")

# Filter by state
success_sessions = store.list_sessions(state="success")
print(f"Successful sessions: {len(success_sessions)}")

In [ ]:
# Load a full session
first_session_id = all_sessions[0].session_id
full_session = store.load_session(first_session_id)

print("Tactic name :", full_session.tactic_name)
print("Tactic path :", full_session.tactic_path)
print("State       :", full_session.state)
print("Delivery    :", str(full_session.delivery)[:80])

for agent_name, calls in full_session.agent_sessions.items():
    print(f"\nAgent '{agent_name}': {len(calls)} call(s)")
    for call in calls:
        print(f"  interrupts={len(call.interrupts)}  retries={call.exception_retries_count}  cost={call.cost}")

---
## Section 12 — Cost Aggregation and Debugging Failures

The `LogStore` lets you aggregate costs across many sessions and inspect full tracebacks for failures.

In [ ]:
# Aggregate cost across all classifier sessions
cost_summary = store.export_cost_summary(
    tactic_path=f"::{ClassifierTactic.name}",  # stable tactic path filter
)
print("Cost summary:")
print(json.dumps(cost_summary, indent=2, default=str))

In [ ]:
# Simulate a failure so we can inspect its traceback
class FailingTactic(Tactic):
    name = "failing_demo"
    agent_group = ["agent"]

    def call(self, task: str, **kwargs) -> str:
        if "fail" in task.lower():
            raise ValueError(f"Intentional failure triggered by input: '{task}'")
        agent = self.agents["agent"]
        agent.open("ok")
        agent.receive(task)
        return agent.respond().content


failing_config = {
    "global": {"model_name": DEFAULT_MODEL},
    "agent_configs": [{"name": "agent", "system_prompt": "You are helpful."}],
}

failing_tactic = FailingTactic(failing_config, log_store=store)

try:
    failing_tactic("please fail now")
except Exception as e:
    print(f"Caught: {e}")

# Find the failed session and inspect its traceback
failed_sessions = store.list_sessions(state="failure", limit=1)
if failed_sessions:
    record = store.load_session_record(failed_sessions[0].session_id)
    print("\nFailed session traceback:")
    print(record.traceback[:500] if record.traceback else "(no traceback stored)")

In [ ]:
# Export a session as JSON for offline analysis
json_dump = store.export_session(all_sessions[0].session_id, format="json")
session_data = json.loads(json_dump)
print("Session JSON keys:", list(session_data.keys()) if isinstance(session_data, dict) else type(session_data))

In [ ]:
# Terminal logging for development — pretty-prints with colors
from lllm.logging import setup_logging
setup_logging(level="INFO")
print("Colored logging enabled. Future calls will print to stderr.")

---
## Summary

| Concept | Key API |
|---------|----------|
| Define a tactic | `class MyTactic(Tactic): name=...; agent_group=[...]` |
| Implement logic | `def call(self, task, **kwargs) -> str` |
| Access agents | `self.agents["agent_name"]` |
| Call a tactic | `tactic(task)` |
| Get session | `tactic(task, return_session=True)` |
| Tag a call | `tactic(task, tags={"env": "prod"})` |
| Batch calls | `tactic.bcall(tasks, max_workers=N)` |
| Async call | `await tactic.acall(task)` |
| Streaming async | `async for idx, r in tactic.ccall(tasks)` |
| Compose tactics | `self.sub = OtherTactic(config)` in `__init__` |
| Extend a tactic | `class Child(Parent): name=...; agent_group=[...]` |
| Unregistered base | `class Base(Tactic, register=False)` |
| SQLite log store | `sqlite_store("./runs.db")` + `Tactic(config, log_store=store)` |
| List sessions | `store.list_sessions(tags=..., state=...)` |
| Load full session | `store.load_session(session_id)` |
| Cost summary | `store.export_cost_summary(tactic_path=...)` |

**Next:** [03_advanced_features.ipynb](03_advanced_features.ipynb) — tools, proxies, config discovery, skills, streaming, and more.